# 02 Wirkung der Begehungen

Teil 2: Begehungen und Wirkung

**Worum es geht**
- Frage: Hat sich das Betriebsverhalten einer Station nach einer Begehung messbar geändert?
- Und passt die Änderung zur dokumentierten Änderungs-Kategorie (keine, geringe, mittlere, viele)?
- Die Begehung ist hier ein Ereignis („was hat sich danach geändert?“), kein Label („hat die Station einen Fehler?“).
- Grund: Die bisherige Zielspalte `n_actions > 0` ist bei 76 % der Stationen wahr und trennt deshalb nichts. Die Zielgröße kommt hier aus den Messdaten selbst.

**Daten**
- liest: Begehungstabelle `Ergebnis_Optimierung` (133 Spalten), `Zuordnung.xlsx` (Adresse zu Zählernummer), stündliche Zeitreihen aus `data/processed/real_meters/`
- schreibt: `data/processed/begehungen_aufbereitet.csv`, `data/processed/begehungs_wirkung.csv` mit Lauf-Kontext `begehungs_wirkung.json`
- vorher ausführen: Notebook 00

**Code**
- Logik in `src/load_begehungen.py`, Tests in `tests/test_load_begehungen.py`

## 1 Setup und Daten laden

- Bibliotheken laden, Projektwurzel finden, verfügbare Zeitreihen zählen.

In [1]:
import os as _os
from pathlib import Path as _Path
#Arbeitsordner ist immer notebooks/, auch aus den Teil-Ordnern
_os.chdir(next(p for p in [_Path.cwd(), *_Path.cwd().parents] if (p / "src").is_dir()) / "notebooks")
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.load_begehungen import begehungen_mit_zaehler, meter_changes, parallel_meters, visit_hours

warnings.filterwarnings("ignore", message="Data Validation extension", module="openpyxl")
pd.set_option("display.max_columns", 40)
plt.rcParams["figure.facecolor"] = "white"

METER_DIR = ROOT / "data" / "processed" / "real_meters"
VERFUEGBAR = {int(p.stem) for p in METER_DIR.glob("*.csv")}
print(f"{len(VERFUEGBAR)} Zeitreihen in {METER_DIR.relative_to(ROOT)}")

SPALTEN = {
    "Volume flow (l/h)": "flow", "Return temperature (°C)": "rt",
    "Flow temperature (°C)": "vl", "Power (kW)": "kw",
    "Temperature difference (°C)": "dt",
}


def zeitreihe(meter):
    """Stundenreihe eines Zaehlers mit kurzen Spaltennamen."""
    pfad = METER_DIR / f"{int(meter)}.csv"
    if not pfad.is_file():
        return None
    df = pd.read_csv(pfad, parse_dates=["Timestamp"])
    return df.set_index("Timestamp").sort_index().rename(columns=SPALTEN)

93 Zeitreihen in data/processed/real_meters


- Das Notebook sucht die Projektwurzel selbst (nach oben, bis `src/` auftaucht). Es läuft deshalb aus `notebooks/` und aus dem Hauptordner.
- Der Loader liegt in `src/load_begehungen.py`, nicht im Notebook: Die Zuordnung Adresse zu Zählernummer ist Datenzugriff und wird auch von späteren Auswertungen gebraucht.
- `VERFUEGBAR` sind die Zähler mit Zeitreihe (hier 93). Damit löst der Loader Adressen mit Zählerwechsel auf ein Gerät mit Daten auf.
- `zeitreihe()` liest eine Reihe auf Stundenraster ein, mit kurzen Spaltennamen: `rt` Rücklauf, `flow` Durchfluss, `dt` Spreizung (Vorlauf minus Rücklauf).

## 2 Begehungen und Zählernummern

- Jede Begehung bekommt über `Zuordnung.xlsx` eine Zählernummer, nicht über `Nodes_Edges.ods`.
- Die aufbereitete Begehungstabelle wird zum Nachschlagen exportiert.

In [2]:
beg = begehungen_mit_zaehler(available=VERFUEGBAR)

print(f"Begehungen in der Excel : {len(beg)}")
print(f"  mit Zaehlernummer     : {beg['meter'].notna().sum()}")
print(f"  mit Datum             : {beg['datum'].notna().sum()}")
print(f"  mit Uhrzeit           : {beg['von'].notna().sum()}")
print(f"  mit Heizzeit-Aenderung: {int(beg['heizzeit_geaendert'].sum())}")

print("\nAenderungs-Kategorie:")
display(beg["kategorie"].value_counts(dropna=False).to_frame("n"))

wechsel = meter_changes()
print(f"Adressen mit Zaehlerwechsel ueber die Jahre: {len(wechsel)}")
for zaehler in wechsel.values():
    print(f"  {zaehler}")
parallel = parallel_meters()
print(f"Adressen mit mehreren Zaehlern im selben Jahr (kein Wechsel): {len(parallel)}")
for zaehler in parallel.values():
    print(f"  {zaehler}")

beg_export = ROOT / "data" / "processed" / "begehungen_aufbereitet.csv"
beg.to_csv(beg_export, index=False, sep=";", decimal=",", encoding="utf-8-sig")
print(f"\nBegehungstabelle ({beg.shape[0]} Zeilen x {beg.shape[1]} Spalten) -> {beg_export.name}")

Begehungen in der Excel : 66
  mit Zaehlernummer     : 66
  mit Datum             : 58
  mit Uhrzeit           : 37
  mit Heizzeit-Aenderung: 8

Aenderungs-Kategorie:


,n
kategorie,
mittlere,36
viele,21
geringe,5
keine,4


Adressen mit Zaehlerwechsel ueber die Jahre: 2
  [65273846, 80912176]
  [58258552, 80912192]
Adressen mit mehreren Zaehlern im selben Jahr (kein Wechsel): 3
  [81050517, 81062289]
  [80912161, 80912177]
  [80912166, 80912175]

Begehungstabelle (66 Zeilen x 26 Spalten) -> begehungen_aufbereitet.csv


- Alle 66 Begehungen bekommen eine Zählernummer. Über die Adressen aus `Nodes_Edges.ods` (Grundlage von `hast_features.csv`) waren es nur 54; dort fehlen 16 Adressen, darunter die Zähler 72167790, 68956340, 80912200 und 80912169. Der Loader korrigiert außerdem einen Tippfehler in einem Straßennamen, der allein vier Zuordnungen kostete.
- Neu nutzbar ist die Änderungs-Kategorie. `inspections_tidy.csv` hatte nur 21 der 133 Spalten übernommen, diese nicht. Sie ist ein vierstufiges, fachlich vergebenes Label und damit besser als `n_actions > 0` mit 76 % Basisrate.
- Fünf Adressen haben mehr als eine Zählernummer:
  - zwei echte Zählerwechsel (z. B. 65273846 zu 80912176 ab 2025), die Daten dazu sind laut Versorger verloren; `meter_changes()` liefert nur diese
  - drei mit zwei Zählern im selben Jahresblatt (`parallel_meters()`): einmal zwei Übergabestationen auf einem Grundstück, zweimal ein Zähler 2024 unter falscher Adresse geführt, ab 2025 korrigiert
  - ein Vorher/Nachher über einen echten Wechsel vergleicht zwei Geräte, nicht zwei Zustände einer Station
- Export `begehungen_aufbereitet.csv`: 66 Zeilen mal 26 Spalten, Semikolon und `utf-8-sig`, damit Excel sie per Doppelklick richtig öffnet. Die Tabelle `beg` (eine Zeile je Begehung) nutzen alle folgenden Abschnitte.

## 3 Vorher/Nachher je Station

- Vergleich je Station: 21 Tage vor gegen 21 Tage nach der Begehung.
- Kurz genug, dass sich Wetter und Jahreszeit kaum ändern; lang genug für stabile Mediane.
- Median statt Mittelwert, weil einzelne Zapfspitzen den Mittelwert verziehen.

In [3]:
FENSTER = pd.Timedelta(days=21)
MIN_STUNDEN = 200


def vorher_nachher(meter, datum):
    """Mediane von Ruecklauf und Durchfluss vor/nach dem Stichtag."""
    df = zeitreihe(meter)
    if df is None:
        return None
    vor = df.loc[datum - FENSTER : datum - pd.Timedelta(hours=1)]
    nach = df.loc[datum + pd.Timedelta(days=1) : datum + FENSTER]
    if vor["rt"].count() < MIN_STUNDEN or nach["rt"].count() < MIN_STUNDEN:
        return None
    return {
        "rt_vor": vor["rt"].median(), "rt_nach": nach["rt"].median(),
        "flow_vor": vor["flow"].median(), "flow_nach": nach["flow"].median(),
    }


zeilen = []
for _, r in beg[beg["datum"].notna() & beg["meter"].notna()].iterrows():
    werte = vorher_nachher(int(r["meter"]), r["datum"].normalize())
    if werte is None:
        continue
    zeilen.append({
        "meter": int(r["meter"]), "adresse": r["key"],
        "datum": r["datum"].normalize(), "kategorie": r["kategorie"],
        "von": str(r["von"])[:5] if pd.notna(r["von"]) else "",
        "meter_gewechselt": r["meter_gewechselt"], **werte,
    })

wirkung = pd.DataFrame(zeilen)
wirkung["d_rt"] = wirkung["rt_nach"] - wirkung["rt_vor"]
wirkung["d_flow"] = wirkung["flow_nach"] - wirkung["flow_vor"]
print(f"auswertbar: {len(wirkung)} von {beg['datum'].notna().sum()} Begehungen mit Datum")

top = (wirkung.reindex(wirkung["d_rt"].abs().sort_values(ascending=False).index)
       .head(8)[["meter", "datum", "kategorie", "rt_vor", "rt_nach",
                 "d_rt", "flow_vor", "flow_nach"]])
display(top.style.format({c: "{:.1f}" for c in top.select_dtypes("number")}))

auswertbar: 44 von 58 Begehungen mit Datum


,meter,datum,kategorie,rt_vor,rt_nach,d_rt,flow_vor,flow_nach
8,72167783.0,2025-09-11 00:00:00,viele,18.6,71.0,52.4,0.0,134.0
30,80912190.0,2024-09-12 00:00:00,mittlere,70.1,25.3,-44.8,117.0,11.0
9,68956351.0,2024-09-09 00:00:00,viele,79.6,49.5,-30.1,499.0,0.0
32,68956372.0,2024-10-14 00:00:00,viele,71.7,53.6,-18.1,358.0,90.0
38,80912169.0,2024-11-19 00:00:00,geringe,44.2,59.7,15.5,128.5,219.0
3,80912172.0,2024-09-10 00:00:00,viele,26.1,40.1,14.0,0.0,0.0
6,72135486.0,2024-09-11 00:00:00,viele,48.7,34.9,-13.8,0.0,6.0
23,80912173.0,2024-09-10 00:00:00,mittlere,54.0,67.3,13.3,44.0,196.0


- Je Fenster braucht es mindestens 200 gemessene Stunden (`MIN_STUNDEN`), sonst fällt die Begehung heraus. Der Begehungstag selbst gehört zu keinem Fenster, weil dort eingegriffen wird.
- Auswertbar sind 44 von 58 Begehungen mit Datum. `d_rt` und `d_flow` sind jeweils nachher minus vorher; negativ heißt gesunken, beim Rücklauf also besser. `meter_gewechselt` markiert Fälle, in denen zwei Geräte verglichen werden könnten. Die Tabelle zeigt die 8 größten Rücklaufänderungen nach Betrag.
- Warum nicht Winter gegen Winter: Ein Jahresabstand bringt anderes Wetter und Regression zur Mitte. Stationen mit hohem Ausgangswert fallen ohnehin (im Winter-Vergleich r = −0,55 zwischen Ausgangswert und Änderung). Im engen Fenster ist beides klein.
- Einzelfälle sagen hier mehr als der Mittelwert: Zähler 68956351 fällt von 499 auf 0 l/h. Ein Ventil stand drei Jahre offen und wurde am Begehungstag geschlossen. In `inspections_tidy.csv` steht dazu `n_actions = 0`.

## 4 Vierfeldertafel: Zustand vor und nach der Begehung

- Frage aus dem Meeting: Was, wenn der Zustand vor und nach der Begehung schlecht war? Dafür braucht es eine Schwelle je Fenster, nicht nur die Differenz.
- Schwelle: Rücklauf-Median über 50 °C = schlechte Auskühlung (das Wasser kühlt in der Station zu wenig ab). Dieselbe Schwelle nutzt das Merkmal `rt_above_50_share_loaded` in `src/features.py`.
- Jede auswertbare Begehung fällt in eine von vier Zellen: gut zu gut, gut zu schlecht, schlecht zu gut, schlecht zu schlecht.

In [4]:
RT_SCHWELLE = 50.0

wirkung["zustand_vor"] = np.where(wirkung["rt_vor"] > RT_SCHWELLE, "schlecht", "gut")
wirkung["zustand_nach"] = np.where(wirkung["rt_nach"] > RT_SCHWELLE, "schlecht", "gut")

vierfeld = pd.crosstab(wirkung["zustand_vor"], wirkung["zustand_nach"],
                        rownames=["vorher"], colnames=["nachher"])
vierfeld = vierfeld.reindex(index=["gut", "schlecht"], columns=["gut", "schlecht"], fill_value=0)
print(f"Vierfeldertafel (n={len(wirkung)}, Schwelle {RT_SCHWELLE:.0f} °C Ruecklauf):")
display(vierfeld)

anteil = (vierfeld / vierfeld.values.sum() * 100).round(1)
print("\nAnteile in %:")
display(anteil)

schlecht_schlecht = wirkung[(wirkung["zustand_vor"] == "schlecht")
                            & (wirkung["zustand_nach"] == "schlecht")]
print(f"\nschlecht -> schlecht: {len(schlecht_schlecht)} von {len(wirkung)} Begehungen "
      f"({len(schlecht_schlecht) / len(wirkung) * 100:.0f} %) - Begehung ohne messbare Wirkung")
if len(schlecht_schlecht):
    display(schlecht_schlecht[["meter", "datum", "kategorie",
                                "rt_vor", "rt_nach", "d_rt"]]
            .sort_values("rt_nach", ascending=False)
            .style.format({c: "{:.1f}" for c in ["rt_vor", "rt_nach", "d_rt"]}))

Vierfeldertafel (n=44, Schwelle 50 °C Ruecklauf):


nachher,gut,schlecht
vorher,,
gut,27,2
schlecht,4,11



Anteile in %:


nachher,gut,schlecht
vorher,,
gut,61.4,4.5
schlecht,9.1,25.0



schlecht -> schlecht: 11 von 44 Begehungen (25 %) - Begehung ohne messbare Wirkung


,meter,datum,kategorie,rt_vor,rt_nach,d_rt
22,80912182,2024-09-19 00:00:00,mittlere,78.5,75.5,-3.0
42,80912154,2024-09-23 00:00:00,mittlere,79.6,74.0,-5.6
23,80912173,2024-09-10 00:00:00,mittlere,54.0,67.3,13.3
4,68956352,2024-09-18 00:00:00,mittlere,58.9,58.8,-0.1
32,68956372,2024-10-14 00:00:00,viele,71.7,53.6,-18.1
35,80912178,2024-09-23 00:00:00,keine,53.6,53.4,-0.2
43,68956355,2024-09-16 00:00:00,viele,59.3,53.0,-6.3
39,80912170,2024-11-19 00:00:00,mittlere,53.3,52.1,-1.2
18,68956373,2025-03-12 00:00:00,mittlere,57.8,51.4,-6.4
33,68956344,2024-09-20 00:00:00,viele,50.4,50.4,0.0


- In der Tafel stehen 44 Begehungen: 27 gut zu gut (61,4 %), 2 gut zu schlecht (4,5 %), 4 schlecht zu gut (9,1 %), 11 schlecht zu schlecht (25,0 %).
- Schlecht zu schlecht ist fachlich der wichtigste Fall: begangen, aber weiter über der Schwelle. Entweder war die Maßnahme nicht wirksam, oder das dokumentierte Problem war nicht der Grund für die schlechte Auskühlung. Diese Stationen sind Kandidaten für eine zweite Begehung.
- Die feste Schwelle ist gröber als die Differenz-von-Differenzen in Abschnitt 5. Sie sagt nicht, ob eine Verbesserung mit der Begehung zusammenhängt oder ohnehin eingetreten wäre.
- Kleine Fallzahlen je Zelle, deshalb keine Signifikanztests, nur Beschreibung.

## 5 Kontrollgruppe und Differenz-von-Differenzen

- Die rohe Änderung enthält alles, was gleichzeitig passierte, vor allem den Beginn der Heizperiode. Die meisten Begehungen lagen im September.
- Kontrollgruppe: Stationen, die nie begangen wurden, im selben Zeitfenster.
- Differenz-von-Differenzen (DiD) = Änderung der begangenen Station minus Änderung der Kontrollgruppe. Übrig bleibt der Teil, den Jahreszeit, Wetter und Netzfahrweise nicht erklären.

In [5]:
#alle begangenen Stationen raus, auch ohne Datum oder ohne auswertbares Fenster
begangen = set(beg["meter"].dropna().astype(int))
kontrollgruppe = sorted(VERFUEGBAR - begangen)
print(f"Kontrollgruppe (nie begangen): {len(kontrollgruppe)} Stationen")


def kontroll_delta(datum):
    """Median-Ruecklaufaenderung der nie begangenen Stationen im selben Fenster."""
    werte = []
    for meter in kontrollgruppe:
        w = vorher_nachher(meter, datum)
        if w is not None:
            werte.append(w["rt_nach"] - w["rt_vor"])
    return np.median(werte) if werte else np.nan


delta_je_datum = {d: kontroll_delta(pd.Timestamp(d)) for d in sorted(wirkung["datum"].unique())}
wirkung["kontrolle"] = wirkung["datum"].map(delta_je_datum)
wirkung["did"] = wirkung["d_rt"] - wirkung["kontrolle"]

REIHENFOLGE = ["keine", "geringe", "mittlere", "viele"]
tabelle = (wirkung.groupby("kategorie")
           .agg(n=("did", "size"), roh=("d_rt", "median"),
                kontrolle=("kontrolle", "median"), DiD=("did", "median"))
           .reindex(REIHENFOLGE).round(2))
display(tabelle)

for kat in ["mittlere", "viele"]:
    werte = wirkung.loc[wirkung["kategorie"] == kat, "did"].dropna()
    if len(werte) > 5:
        p = stats.wilcoxon(werte)[1]
        print(f"{kat:9s} n={len(werte):2d}  Median {werte.median():+.2f} K  Wilcoxon p = {p:.3f}")

Kontrollgruppe (nie begangen): 37 Stationen


,n,roh,kontrolle,DiD
kategorie,,,,
keine,2,5.10,-1.85,6.95
geringe,3,1.50,-1.40,3.10
mittlere,25,-1.50,-0.90,-1.56
viele,14,-3.82,-0.55,-3.05


mittlere  n=24  Median -1.56 K  Wilcoxon p = 0.252
viele     n=13  Median -3.05 K  Wilcoxon p = 0.191


- Kontrollgruppe: 37 nie begangene Stationen. Begangene Stationen fallen ganz heraus, auch wenn ihre Begehung kein Datum hat oder hier nicht auswertbar ist. Ihre mediane Rücklaufänderung wird je Begehungsdatum einmal berechnet und an jede Begehung mit diesem Datum gehängt.
- Für die zwei Begehungen vom September 2025 gibt es keinen Kontrollwert: Nach dem 11.07.2025 liefern nur noch die fünf Nachlieferungszähler Daten, und alle fünf wurden begangen. Der DiD bei „mittlere“ und „viele“ stützt sich deshalb auf 24 und 13 Begehungen.
- DiD je Kategorie: keine +6,95 K, geringe +3,10 K, mittlere −1,56 K, viele −3,05 K. Das ist monoton über alle vier Stufen: Je mehr laut Excel geändert wurde, desto stärker sinkt der Rücklauf gegenüber den unbehandelten Stationen.
- Nicht signifikant: Wilcoxon-Test (prüft, ob die Werte im Median von null abweichen) p = 0,25 für „mittlere“, p = 0,19 für „viele“. Gründe:
  - „keine“ und „geringe“ haben nur 2 bzw. 3 Fälle
  - begangen wurden vermutlich eher auffällige Stationen, die ohnehin stärker fallen; diesen Auswahleffekt fängt DiD nicht ab
- Rechenweise: Die Differenz wird je Station gebildet, danach der Median, weil jede Station ihr eigenes Datum hat. Deshalb ergibt −3,82 minus −0,55 in der Tabelle nicht genau −3,05.

## 6 Ist die Begehung in den Daten sichtbar?

- Öffnet jemand die Station, läuft heißes Wasser durch, ohne Wärme abzugeben: Vorlauf hoch, Rücklauf fast genauso hoch, Spreizung (ΔT) nahe null.
- Gesucht wird das kleinste ΔT im dokumentierten Zeitfenster der Begehung (Uhrzeit von/bis aus der Excel, plus eine Stunde Puffer). Unter 3 K gilt als Treffer.
- Zweck: Kontrolle, ob Begehungsdatum und Zählerzuordnung stimmen.

In [6]:
DT_SCHWELLE = 3.0   #K


def signatur(meter, datum, stunden):
    """Kleinstes ΔT im dokumentierten Begehungsfenster."""
    df = zeitreihe(meter)
    if df is None or stunden is None:
        return np.nan
    start, ende = stunden
    fenster = df.loc[datum + pd.Timedelta(hours=start) : datum + pd.Timedelta(hours=ende + 1)]
    return fenster["dt"].min() if fenster["dt"].notna().any() else np.nan


mit_zeit = beg[beg["datum"].notna() & beg["meter"].notna() & beg["von"].notna()].copy()
mit_zeit["dt_min"] = [
    signatur(int(r["meter"]), r["datum"].normalize(), visit_hours(r))
    for _, r in mit_zeit.iterrows()
]
auswertbar = mit_zeit[mit_zeit["dt_min"].notna()]
treffer = auswertbar["dt_min"] < DT_SCHWELLE
print(f"Begehungen mit Uhrzeit und Daten: {len(auswertbar)}")
print(f"davon ΔT < {DT_SCHWELLE:.0f} K im Zeitfenster: {treffer.sum()} ({treffer.mean():.0%})")

sig = (auswertbar.loc[treffer, ["meter", "datum", "von", "bis", "kategorie", "dt_min"]]
       .sort_values("dt_min"))
display(sig.style.format({"dt_min": "{:.2f}"}))

Begehungen mit Uhrzeit und Daten: 25
davon ΔT < 3 K im Zeitfenster: 12 (48%)


,meter,datum,von,bis,kategorie,dt_min
20,80912159,2024-09-10 00:00:00,13:30:00,14:41:00,mittlere,-1.60
7,68956371,2025-09-16 00:00:00,14:15:00,14:59:00,mittlere,0.00
18,72167783,2025-09-11 00:00:00,08:40:00,09:58:00,viele,0.10
19,68956351,2024-09-09 00:00:00,15:00:00,16:37:00,viele,0.10
32,80912163,2024-08-26 00:00:00,16:10:00,17:05:00,viele,0.10
55,80912178,2024-09-23 00:00:00,12:00:00,12:58:00,keine,0.50
65,68956355,2024-09-16 00:00:00,12:50:00,14:10:00,viele,0.80
2,68956348,2024-09-09 00:00:00,08:30:00,09:55:00,viele,1.30
3,68956353,2024-09-23 00:00:00,14:20:00,15:07:00,viele,2.10
5,80912172,2024-09-10 00:00:00,10:30:00,11:20:00,viele,2.10


- 37 Begehungen haben Datum, Zählernummer und Uhrzeit, 25 davon Messdaten im Zeitfenster.
- 12 von 25 (48 %) zeigen die Signatur. Die Tabelle listet sie, die deutlichsten zuerst.
- Das passt dazu, dass Begehungsdatum und Zuordnung stimmen. Ohne diese Kontrolle wüsste man bei fehlendem Effekt nicht, ob nichts passiert ist oder ob man am falschen Zähler sucht.
- Grenze: Bei Stationen mit dauernd laufender Zirkulation kommt niedriges ΔT auch im Normalbetrieb vor. Die Signatur ist eine Bestätigung, kein Beweis.

## 7 Dokumentierte Heizzeit-Änderungen prüfen

- Bei 8 Stationen steht in der Excel, wie die Heizzeit vorher und nachher eingestellt war (z. B. 06:00–22:00 zu 07:00–21:00). Das ist eine Vorhersage, die vor dem Blick in die Daten feststeht.
- Die Tagesstunden werden geteilt: weggefallen (vorher geheizt, jetzt nicht), unverändert (in beiden Plänen), dazugekommen.
- Verglichen wird der typische Durchfluss je Tagesstunde im Winter 2023/24 gegen 2024/25, je Gruppe.
- „Unverändert“ ist die Kontrolle: Fallen alle Gruppen gleich, passt das zum Wetter. Fällt nur „weggefallen“, passt das zur Umstellung.

In [7]:
import re

WINTER_VOR = ("2023-12-01", "2024-02-29")
WINTER_NACH = ("2024-12-01", "2025-02-28")


def fensterstunden(von, bis):
    """Menge der Tagesstunden im Heizfenster, auch ueber Mitternacht."""
    def stunde(v):
        m = re.search(r"(\d{1,2}):(\d{2})", str(v))
        return int(m.group(1)) if m else None
    a, b = stunde(von), stunde(bis)
    if a is None or b is None:
        return None
    return set(range(a, b + 1)) if a <= b else set(range(a, 24)) | set(range(0, b + 1))


def tagesgang(meter, zeitraum, groesse="flow"):
    """Median je Tagesstunde im angegebenen Zeitraum."""
    df = zeitreihe(meter)
    if df is None:
        return None
    w = df.loc[zeitraum[0]:zeitraum[1]]
    return w.groupby(w.index.hour)[groesse].median() if len(w) else None


hz = beg[beg["heizzeit_geaendert"] & beg["meter"].notna()].copy()
print(f"Stationen mit dokumentierter Heizzeit-Aenderung: {len(hz)}\n")
print(f"{'Zaehler':>9}  {'alt':>12} {'neu':>12} | "
      f"{'weggefallen':>11} {'unveraendert':>12} {'dazugekommen':>12}")

hz_zeilen = []
for _, r in hz.iterrows():
    alt = fensterstunden(r["heizzeit_alt_von"], r["heizzeit_alt_bis"])
    neu = fensterstunden(r["heizzeit_neu_von"], r["heizzeit_neu_bis"])
    vor = tagesgang(int(r["meter"]), WINTER_VOR)
    nach = tagesgang(int(r["meter"]), WINTER_NACH)
    if not alt or not neu or vor is None or nach is None or vor.empty or nach.empty:
        continue
    aenderung = (nach - vor).dropna()

    def median_ueber(stunden_menge):
        werte = [aenderung[h] for h in stunden_menge if h in aenderung.index]
        return np.median(werte) if werte else np.nan

    weg, dazu, bleibt = median_ueber(alt - neu), median_ueber(neu - alt), median_ueber(alt & neu)
    hz_zeilen.append({"meter": int(r["meter"]), "adresse": r["key"],
                      "weggefallen": weg, "unveraendert": bleibt, "dazugekommen": dazu})
    a = f"{str(r['heizzeit_alt_von'])[:5]}-{str(r['heizzeit_alt_bis'])[:5]}"
    b = f"{str(r['heizzeit_neu_von'])[:5]}-{str(r['heizzeit_neu_bis'])[:5]}"
    print(f"{int(r['meter']):>9}  {a:>12} {b:>12} | "
          f"{weg:11.0f} {bleibt:12.0f} {dazu:12.0f}")

hz_tab = pd.DataFrame(hz_zeilen)
mit_wegfall = hz_tab.dropna(subset=["weggefallen"])
bestaetigt = mit_wegfall["weggefallen"] < mit_wegfall["unveraendert"]
print(f"\nStationen mit weggefallenen Heizstunden: {len(mit_wegfall)}")
print(f"davon Durchfluss dort staerker gefallen als in unveraenderten Stunden: "
      f"{bestaetigt.sum()} von {len(mit_wegfall)}")

Stationen mit dokumentierter Heizzeit-Aenderung: 8

  Zaehler           alt          neu | weggefallen unveraendert dazugekommen
 80912195   07:00-22:30  02:00-23:59 |         nan           30           23
 80912159   05:00-21:00  07:00-22:00 |         -12          -24          -92
 80912160   04:15-18:00  03:45-20:00 |         nan           68           28
 72167784   06:00-22:00  06:00-22:30 |         nan           46          nan
 72135442   05:30-22:00  05:30-21:00 |        -333            8          nan
 80912156   06:00-22:00  07:00-21:00 |         -40            4          nan
 80912161   06:00-22:00  07:00-22:00 |        -112            0          nan
 68956345   04:00-22:15  06:00-22:00 |         -21           23          nan

Stationen mit weggefallenen Heizstunden: 5
davon Durchfluss dort staerker gefallen als in unveraenderten Stunden: 4 von 5


- Bei 5 Stationen sind Heizstunden weggefallen. Bei 4 davon fiel der Durchfluss dort stärker als in den unveränderten Stunden.
- Beispiel 80912156: weggefallene Stunden −40 l/h, unveränderte +4 l/h.
- Bei dazugekommenen Stunden zeigt sich nichts. Eine längere Heizzeit erzwingt kein Heizen; das entscheidet der Thermostat im Haus.

## 8 Ergebnis speichern

- Schreibt die Wirkungstabelle und den Lauf-Kontext nach `data/processed/`.

In [8]:
import json

ausgabe = ROOT / "data" / "processed" / "begehungs_wirkung.csv"
spalten = ["meter", "adresse", "datum", "kategorie", "von", "meter_gewechselt",
           "rt_vor", "rt_nach", "d_rt", "flow_vor", "flow_nach", "d_flow",
           "kontrolle", "did"]
wirkung[spalten].to_csv(ausgabe, index=False)

meta = {
    "erzeugt": pd.Timestamp.now().isoformat(timespec="seconds"),
    "n_begehungen_excel": int(len(beg)),
    "n_auswertbar": int(len(wirkung)),
    "n_kontrollgruppe": len(kontrollgruppe),
    "fenster_tage": int(FENSTER.days),
    "min_stunden": MIN_STUNDEN,
    "did_je_kategorie": {k: (None if pd.isna(v) else round(float(v), 2))
                         for k, v in tabelle["DiD"].items()},
}
ausgabe.with_suffix(".json").write_text(
    json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"Wirkungstabelle -> {ausgabe.relative_to(ROOT)}")
print(f"Lauf-Kontext    -> {ausgabe.with_suffix('.json').relative_to(ROOT)}")

Wirkungstabelle -> data/processed/begehungs_wirkung.csv
Lauf-Kontext    -> data/processed/begehungs_wirkung.json


- `begehungs_wirkung.csv`: eine Zeile je auswertbarer Begehung mit Rohänderung, Kontrollwert und DiD. `meter_gewechselt` ist dabei, weil der Vergleich dort zwei Geräte betreffen kann.
- `begehungs_wirkung.json`: Fenstergröße, Mindeststunden, Größe der Kontrollgruppe, DiD je Kategorie.
- Die CSV enthält Zählernummern und Adressen. `data/processed/` ist gitignored und bleibt es (siehe Datenschutz in [`docs/installation/vorbereitung.md`](../../docs/installation/vorbereitung.md#datenschutz)).

## Zusammenfassung & Bewertung

**Ergebnis**
- Zuordnung: 66 von 66 Begehungen über `Zuordnung.xlsx` mit Zählernummer (über `Nodes_Edges.ods` nur 54).
- Rücklaufänderung gegenüber der Kontrollgruppe (DiD) monoton über die vier Änderungs-Kategorien, „viele“ am stärksten (−3,05 K).
- Vierfeldertafel (Rücklauf über 50 °C = schlecht, n = 44): 61 % gut zu gut, 25 % schlecht zu schlecht, 9 % schlecht zu gut, 5 % gut zu schlecht. 11 Stationen blieben trotz Begehung über der Schwelle, 2 verschlechterten sich.
- Dokumentierte Heizzeit-Änderung: in 4 von 5 Fällen mit weggefallenen Heizstunden sichtbar.

**Belastbar**
- Begehung im Zeitfenster sichtbar (ΔT unter 3 K) bei 12 von 25 Begehungen mit Uhrzeit, also rund der Hälfte.
- Heizzeit-Test mit eigener Kontrolle (unveränderte Stunden): 4 von 5.
- Einzelfall 68956351: Ein Ventil stand drei Jahre offen (500 bis 950 l/h durchgehend bei 0,1 kW) und wurde am 09.09.2024 geschlossen, über Nacht von 499 auf 0 l/h. In der Begehungstabelle steht dazu keine Maßnahme.
- Das passt dazu, dass Clustering ([Notebook 07](../3_clustering/07_stationscluster.ipynb), Silhouette unter 0,25) und der frühere ML-Klassifikator kein Signal fanden: Ihr Label `n_actions > 0` ist bei 76 % der Stationen wahr und kennt selbst diesen Fall nicht.

**Schwach**
- Die Monotonie ist nicht signifikant (Wilcoxon p = 0,25 bzw. 0,19); „keine“ und „geringe“ haben nur 2 bzw. 3 Fälle.
- Begangen wurden vermutlich eher auffällige Stationen. DiD fängt diesen Auswahleffekt nicht ab, deshalb nur „passt zu einer Wirkung“.
- Die Vierfeldertafel ist reine Beschreibung ohne Test.

**Offen**
- Zählerwechsel und Mehrfachzähler bei fünf Adressen: Vorher/Nachher kann dort zwei Geräte vergleichen.
- Zähler 72167783: Rücklauf nach der Begehung von 18,6 auf 71,0 °C. Inbetriebnahme oder Fehler? Vor jeder Mittelwertbildung klären.